# Data Science Toolkit
A step-by-step notebook for exploring **any CSV dataset**. All calculations live in [toolkit.py](toolkit.py); this notebook only chooses *what* to look at.

**How to use it:** go through the steps from top to bottom. Each step has a short explanation and a cell that starts with its own settings in `CAPITALS`. Change them if you want, then run the cell.

Column names to pick from are printed in **Step 2.4**.

## Step 1 - Setup
### 1.1 Load the toolkit

In [58]:
%load_ext autoreload
%autoreload 2
import toolkit as tk

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1.2 Available datasets
Put your CSV files in the `data/` folder. This lists what is there.

In [59]:
DATA_DIR = "data"   # folder that holds the datasets

tk.list_datasets(DATA_DIR)

IRIS.csv                                          0.00 MB
breast_cancer_raw.csv                             0.12 MB
spotify-tracks-dataset-detailed.csv              19.43 MB
wine_data_raw.csv                                 0.01 MB


### 1.3 Load the dataset
Copy one of the file names above into `FILE`. The separator and header row are detected for you.

In [60]:
FILE = "spotify-tracks-dataset-detailed.csv"  # a file from the list above (None = the only .csv in DATA_DIR)

raw = tk.load_csv(DATA_DIR, FILE)
raw.head()

Loaded data/spotify-tracks-dataset-detailed.csv -> 114000 rows x 20 columns (header row: yes)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Step 2 - Data inspection
A first look at the dataset as it was loaded.
### 2.1 Dimensions

In [61]:
tk.shape(raw)

Rows: 114000, Columns: 20


### 2.2 Columns
Data type, number of filled and missing values, and number of unique values per column.

In [62]:
tk.column_info(raw)

,dtype,non_null,missing,unique
track_id,object,114000,0,89741
artists,object,113999,1,31437
album_name,object,113999,1,46589
track_name,object,113999,1,73608
popularity,int64,114000,0,101
duration_ms,int64,114000,0,50697
explicit,bool,114000,0,2
danceability,float64,114000,0,1174
energy,float64,114000,0,2083
key,int64,114000,0,12


### 2.3 Basic statistics
Count, mean, standard deviation, minimum, Q1, median, Q3 and maximum of each numeric column.

In [ ]:
tk.summary(raw)

### 2.4 Choose the columns to work with
Builds the working dataset `data` that every next step uses.
- **TARGET**: the column with the labels/classes (used to color the plots). `"auto"` takes the last column if it looks like labels.
- **FEATURES**: the measurement columns to analyse. `None` = all numeric columns except the target.
- **DROP**: columns to ignore, e.g. an ID column.
- **MISSING**: what to do with empty cells: `"drop"` the row, fill with `"mean"` / `"median"`, or `None`.

In [ ]:
TARGET = "auto"     # "auto", a column name, or None
FEATURES = None     # None = all numeric columns, or a list like ["col_1", "col_2"]
DROP = []           # e.g. ["id"]
MISSING = "drop"    # "drop", "mean", "median" or None

data = tk.prepare(raw, target=TARGET, features=FEATURES, drop=DROP, missing=MISSING)

## Step 3 - Data transformation
Rescales columns so they can be compared with each other. Each method returns a new table; the original `data` is not changed.
### 3.1 Min-max normalization
Rescales every column to the range 0 – 1.

In [ ]:
COLUMNS = None      # None = all features, or a list of columns

tk.transform(data, "minmax", COLUMNS).head()

### 3.2 Standardization (z-score)
Every column gets mean 0 and standard deviation 1.

In [ ]:
COLUMNS = None

tk.transform(data, "standard", COLUMNS).head()

### 3.3 Robust scaling
Centers on the median and divides by the IQR. Less sensitive to outliers than standardization.

In [ ]:
COLUMNS = None

tk.transform(data, "robust", COLUMNS).head()

### 3.4 Log transformation
`log(1 + x)`: compresses large values, useful for skewed columns. Columns with negative values are shifted first.

In [ ]:
COLUMNS = None

tk.transform(data, "log", COLUMNS).head()

## Step 4 - Descriptive analysis
Central tendency (mean, median, mode), spread (std, variance), range and quartiles for the columns you choose.

In [ ]:
COLUMNS = None      # None = all features, or e.g. ["col_1", "col_2", "col_3"]

tk.describe(data, COLUMNS)

## Step 5 - Relationships
Are your 2 variables in love with each other? Find out!

Choose two columns to get their **Pearson** correlation (linear relationship), **Spearman** correlation (monotonic relationship) and **covariance**, plus a scatter plot.

In [ ]:
X = None            # first column, e.g. "col_1" (None = first feature)
Y = None            # second column, e.g. "col_2" (None = second feature)

tk.relationship(data, X, Y)

## Step 6 - Outlier analysis
Find values that are far away from the rest.
- **IQR method**: outside `[Q1 - FACTOR·IQR, Q3 + FACTOR·IQR]` (1.5 is the usual choice).
- **Z-score method**: more than `THRESHOLD` standard deviations from the mean (3 is the usual choice).
### 6.1 Detect outliers

In [ ]:
COLUMNS = None      # None = all features
METHOD = "iqr"      # "iqr" or "zscore"
FACTOR = 1.5        # used by "iqr"
THRESHOLD = 3.0     # used by "zscore"

outlier_table, outlier_rows = tk.outliers(data, COLUMNS, METHOD, FACTOR, THRESHOLD)
outlier_table

### 6.2 Rows with outliers

In [ ]:
N_ROWS = 10         # how many rows to show

outlier_rows.head(N_ROWS)

### 6.3 Visualize outliers
Boxplots on standardized values so all columns share one scale. Red dots are outliers.

In [ ]:
COLUMNS = None
SAVE_AS = None      # None, or a file name like "outliers" to save it in output/

tk.plot_outliers(data, COLUMNS, factor=FACTOR, save_as=SAVE_AS)

## Step 7 - Visualization
Plot the columns you choose.

| `KIND` | Shows | Columns |
|---|---|---|
| `"hist"` | distribution of each column | any number |
| `"box"` | median, quartiles and outliers | any number |
| `"violin"` | distribution shape | any number |
| `"scatter"` | one column against another | exactly 2 |
| `"line"` | values in row order | any number |

Use `SOURCE` to plot transformed values instead of the original ones.

In [ ]:
COLUMNS = None          # None = first 4 features, or e.g. ["col_1", "col_2"] (scatter: exactly 2)
KIND = "hist"           # "hist", "box", "violin", "scatter" or "line"
SOURCE = None           # None = original values, or "minmax", "standard", "robust", "log"
COLOR = "tab:blue"      # any matplotlib color (used when not colored by target)
COLOR_BY = "target"     # "target" = one color per class, a column name, or None
TITLE = None            # None = automatic
SAVE_AS = None          # None, or a file name to save it in output/

plot_data = tk.transform(data, SOURCE, COLUMNS) if SOURCE else data
tk.plot(plot_data, COLUMNS, kind=KIND, color=COLOR, color_by=COLOR_BY, title=TITLE, save_as=SAVE_AS)

## Step 8 - Relationship maps
Relationships between many columns at once. Only the lower half is shown because the matrix is symmetric.
### 8.1 Correlation heatmap
Values from -1 (move in opposite directions) to 1 (move together).

In [ ]:
COLUMNS = None          # None = all features, or a list of at least 2 columns
METHOD = "pearson"      # "pearson" or "spearman"
SAVE_AS = None

tk.correlation_heatmap(data, COLUMNS, METHOD, save_as=SAVE_AS)

### 8.2 Covariance heatmap
Like correlation but in the original units, so columns with large values dominate the colors.

In [ ]:
COLUMNS = None
SAVE_AS = None

tk.covariance_heatmap(data, COLUMNS, save_as=SAVE_AS)

## Step 9 - Principal Component Analysis
When you have a very wide dataset, you can reduce its dimensionality using principal component analysis (PCA). PCA relies on the concepts of linear transformations and singular value decomposition (SVD) in linear algebra.
Luckily, scikit-learn provides a package that we can use to perform PCA instead of worrying about the linear algebra. You can find the documentation [here](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html).

Standardizing first (`SCALE = "standard"`) is recommended, otherwise columns with large values dominate the components.
### 9.1 Run PCA

In [ ]:
COLUMNS = None          # None = all features
N_COMPONENTS = 3        # 2 or 3
SCALE = "standard"      # None, "minmax", "standard", "robust" or "log"

components, loadings = tk.pca(data, COLUMNS, N_COMPONENTS, SCALE)
components.head()

### 9.2 Loadings
How much each original column contributes to each component, sorted by importance for PC1.

In [ ]:
loadings

### 9.3 Plot the components

In [ ]:
COLOR_BY = "target"     # "target" = one color per class, or None
SAVE_AS = None

tk.plot_pca(components, color_by=COLOR_BY, save_as=SAVE_AS)